# Solar Filament Segmentation Challenge 2026 -- ResNet50 BYOL Pretraining (Kaggle)

Domain-specific self-supervised pretraining of a 1-channel ResNet50 on the GONG
H-alpha corpus, via BYOL. Runs `scripts/pretrain_resnet/train_byol.py`, DDP-ready
for Kaggle's 2xT4. Design: `RESNET_PRETRAIN_PLAN.md`. Prerequisite: the
`halpha-preprocessed` Kaggle Dataset (built by `pretrain_gong_kaggle.ipynb`) added
as a notebook input, and a GPU accelerator (ideally 2xT4) enabled.

This is a **multi-session** run -- see section 6 below for the checklist to repeat
each session (mount latest checkpoint, run, version a new checkpoint dataset,
repeat).

### 0. Clone the repo

Requires `jp-pretraining-data-prep` (or wherever this lands) to already be pushed
to `origin`.

In [ ]:
!git clone -b jp-pretraining-data-prep https://github.com/jprakash-1/Solar-Filament-Segmentation.git


In [ ]:
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after the kernel started
!git pull origin jp-pretraining-data-prep


### 1. Install dependencies

Kaggle's GPU images already ship torch/torchvision/pandas/tqdm -- only add what's
missing (`scipy`, for the augmentation's Gaussian blur; `segmentation_models_pytorch`
only if re-running `scripts/pretrain_resnet/model.py`'s `verify_stem_averaging`
check, not needed for training itself).

In [ ]:
!pip install -q scipy


### 2. Mount the preprocessed corpus (+ a checkpoint dataset, on resume sessions)

In the Kaggle UI: **Add Input -> Datasets -> `halpha-preprocessed`** (first
session and every session), and on any session after the first, also add the
checkpoint dataset from the previous session's `/kaggle/working/checkpoints/`
version (see section 6).

Verify the actual mounted layout before trusting a path below -- it depends on how
the dataset was uploaded/zipped, and guessing wrong here just wastes a session:

In [ ]:
!ls /kaggle/input/
!ls /kaggle/input/halpha-preprocessed/ | head


In [ ]:
# Adjust these two if the listing above shows a different layout
# (e.g. files nested one directory deeper than expected).
IMAGES_DIR = "/kaggle/input/halpha-preprocessed"
MANIFEST = "/kaggle/input/halpha-preprocessed/manifest.csv"

# Point at the mounted checkpoint dataset's latest.pt on resume sessions; leave
# as None (the script's own --resume default) for the very first session.
RESUME_CKPT = None  # e.g. "/kaggle/input/halpha-byol-ckpt/latest.pt"


### 3. Smoke test before the real run

Small subset, single-process, 1 epoch -- catches a broken path or a shape bug in
under a minute, before committing a multi-hour session to it. Matches the
"verify against real data before trusting it at scale" standard the Stage 1/2
scripts were held to.

In [ ]:
!python scripts/pretrain_resnet/train_byol.py \
    --images-dir $IMAGES_DIR --manifest $MANIFEST \
    --max-images 200 --epochs 1 --batch-size 16 --num-workers 2 \
    --checkpoint-out /kaggle/working/smoke_ckpt.pt --log-csv /kaggle/working/smoke_log.csv \
    --nn-grid-out /kaggle/working/smoke_nn.png


### 4. Full run: multi-GPU launch

`torchrun --nproc_per_node=2` for Kaggle's 2xT4 (drop to 1 if only a single GPU is
enabled). `NCCL_P2P_DISABLE=1` is the same T4-specific workaround
`PRETRAIN_PLAN.md` section 4.3 already had to apply. `--session-budget-seconds`
should stay well under the account's actual session cap (varies by verification
tier) -- the script self-stops with margin for the final checkpoint write, no need
to watch the clock.

In [ ]:
RESUME_ARGS = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""

!NCCL_P2P_DISABLE=1 torchrun --nproc_per_node=2 scripts/pretrain_resnet/train_byol.py \
    --images-dir $IMAGES_DIR --manifest $MANIFEST \
    --epochs 150 --batch-size 128 --num-workers 4 \
    --checkpoint-out /kaggle/working/checkpoints/latest.pt \
    --log-csv /kaggle/working/checkpoints/loss_log.csv \
    --nn-grid-out /kaggle/working/checkpoints/nn_grid.png \
    --session-budget-seconds 28800 --health-check-every 5 \
    {RESUME_ARGS}


### 5. Sanity-check this session's output

Loss curve, plus the section 8 embedding-std collapse check and nearest-neighbor
visual check that `health_checks.py` already logged during training -- don't
trust the loss curve alone (BYOL can keep dropping loss while representations
collapse).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("/kaggle/working/checkpoints/loss_log.csv")
print(log.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BYOL loss"); axes[0].legend()

health = log.dropna(subset=["embedding_std"])
axes[1].plot(health["epoch"], health["embedding_std"])
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("embedding_std (collapse check)")
plt.tight_layout(); plt.show()


In [ ]:
from PIL import Image
Image.open("/kaggle/working/checkpoints/nn_grid.png")


### 6. Session checklist (repeat each session)

1. Mount `halpha-preprocessed` and the latest `halpha-byol-ckpt` dataset (if any)
   as notebook inputs; set `RESUME_CKPT` above accordingly.
2. Run the smoke test (section 3), then the full launch cell (section 4).
3. The loop self-stops at `--session-budget-seconds`, checkpointing every epoch
   along the way -- an unplanned kill loses at most one epoch.
4. **New Dataset version** from `/kaggle/working/checkpoints/` -> this becomes
   next session's `halpha-byol-ckpt` input.
5. Run section 5's sanity check before ending the session -- catch a collapsed
   run early rather than after several more sessions of wasted compute.
6. Every few sessions: inspect `nn_grid.png` closely (section 5) -- confirm
   neighbors are astronomically similar, not random frames.

**Open items this run should help resolve** (`RESNET_PRETRAIN_PLAN.md` section
11): real images/sec on 2xT4 at this batch size (watch the tqdm it/s from
section 3/4 and back out a per-GPU throughput number), and whether the full
49K-image corpus vs. some thinning is the right call once real wall-clock/epoch
is known.